# Benchmarking Signal- and Vision-Based Deep Learning with SHAP Explainability for Multi-Class 12-Lead ECG Arrhythmia Classification

**Dataset:** PhysioNet/Computing in Cardiology Challenge 2020  
**Models:** 2 Signal-based (S1, S2) · 2 Vision-based (V1, V2) · 1 Hybrid (H1)  
**Explainability:** SHAP  
**Emissions Tracking:** CodeCarbon

---


## 1. Environment Setup and Dependencies

In [ ]:
# 1.1 Install dependencies
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p in ["wfdb", "neurokit2", "PyWavelets", "codecarbon", "ecg-plot", "shap"]:
    try:
        __import__(p.replace("-","_").lower().replace("pywavelets","pywt"))
    except ImportError:
        install(p)
print("Dependencies ready.")


In [ ]:
# 1.2 Imports
import os, sys, json, random, warnings, time, hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from scipy import signal as scipy_signal
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms

import wfdb
import pywt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score)

warnings.filterwarnings('ignore')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")


In [ ]:
# 1.3 Reproducibility and Configuration
@dataclass
class Config:
    # Seeds
    seed: int = 42
    # Paths
    base_dir: str = os.getcwd()
    data_dir: str = ""
    output_dir: str = ""
    cache_dir: str = ""
    # Signal
    target_fs: int = 500
    seq_len: int = 5000  # 10s @ 500Hz
    # Vision
    img_size: int = 224
    # Training
    batch_size: int = 32
    epochs: int = 30
    lr: float = 1e-3
    patience: int = 5
    num_classes: int = 7
    # Splits
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    
    def __post_init__(self):
        self.data_dir = os.path.join(self.base_dir, "physionet.org/files/challenge-2020/1.0.2/training")
        self.output_dir = os.path.join(self.base_dir, "outputs")
        self.cache_dir = os.path.join(self.base_dir, "cache")

CFG = Config()

# Set seeds
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seeds(CFG.seed)

# Create directories
for d in ["outputs/metadata", "outputs/metrics", "outputs/figures/shap",
          "outputs/emissions", "outputs/models", "cache/images"]:
    os.makedirs(os.path.join(CFG.base_dir, d), exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"Config: seed={CFG.seed}, fs={CFG.target_fs}Hz, seq_len={CFG.seq_len}, img={CFG.img_size}x{CFG.img_size}")

CLASS_NAMES = ['NORM', 'AFIB', 'AFLT', '1dAVb', 'RBBB', 'LBBB', 'OTHERS']
NUM_CLASSES = len(CLASS_NAMES)


## 2. Dataset Description

The **PhysioNet/Computing in Cardiology Challenge 2020** dataset contains 12-lead ECG recordings from multiple databases:
- CPSC 2018 (+ extra)
- Georgia 12-Lead ECG Challenge
- PTB & PTB-XL
- St. Petersburg INCART

Source: https://physionet.org/content/challenge-2020/1.0.2/


## 3. Label Mapping Definitions

### SNOMED CT Code Mapping to 7 Classes

| Class | SNOMED Codes | Description |
|-------|-------------|-------------|
| NORM | 426783006 | Normal sinus rhythm |
| AFIB | 164889003, 164890007 | Atrial fibrillation |
| AFLT | 164890007 (overlap handled), 233893006 | Atrial flutter |
| 1dAVb | 270492004 | 1st degree AV block |
| RBBB | 59118001, 713427006 | Right bundle branch block |
| LBBB | 164909002, 713426002 | Left bundle branch block |
| OTHERS | All other abnormal | Other abnormalities |

### Priority Ordering (for multi-label records)
`AFIB > AFLT > 1dAVb > RBBB > LBBB > OTHERS > NORM`


In [ ]:
# 3.1 SNOMED CT to Class Mapping
SNOMED_MAP = {
    # NORM
    '426783006': 'NORM',
    '426177001': 'NORM',  # Sinus bradycardia -> could be NORM variant
    # AFIB
    '164889003': 'AFIB',
    '284470004': 'AFIB',  # PAC (related)
    # AFLT  
    '164890007': 'AFLT',
    '233893006': 'AFLT',
    # 1dAVb
    '270492004': '1dAVb',
    '426627000': '1dAVb',
    # RBBB
    '59118001': 'RBBB',
    '713427006': 'RBBB',
    # LBBB
    '164909002': 'LBBB',
    '713426002': 'LBBB',
    '445118002': 'LBBB',
}

# Priority order (index = priority, lower = higher priority)
PRIORITY = ['AFIB', 'AFLT', '1dAVb', 'RBBB', 'LBBB', 'OTHERS', 'NORM']

def map_labels(snomed_codes: List[str]) -> str:
    """Map list of SNOMED codes to single class using priority ordering."""
    mapped = set()
    for code in snomed_codes:
        code = code.strip()
        if code in SNOMED_MAP:
            mapped.add(SNOMED_MAP[code])
        else:
            mapped.add('OTHERS')
    
    if not mapped:
        return 'NORM'
    
    # Apply priority
    for cls in PRIORITY:
        if cls in mapped:
            return cls
    return 'OTHERS'

print("Label mapping ready. Priority:", ' > '.join(PRIORITY))


## 4. Data Loading and Metadata Inspection

In [ ]:
# 4.1 Scan all records
def find_all_records(data_dir):
    """Find all .hea files in the dataset directory tree."""
    records = []
    for root, dirs, files in os.walk(data_dir):
        for f in files:
            if f.endswith('.hea'):
                rec_path = os.path.join(root, f[:-4])  # remove .hea
                records.append(rec_path)
    return sorted(records)

def parse_header(hea_path):
    """Parse a WFDB header file to extract metadata and diagnosis codes."""
    try:
        record = wfdb.rdheader(hea_path)
        # Extract diagnosis from comments
        dx_codes = []
        for comment in record.comments:
            if comment.startswith('Dx:') or comment.startswith('#Dx:'):
                codes = comment.replace('#Dx:', '').replace('Dx:', '').strip()
                dx_codes = [c.strip() for c in codes.split(',')]
                break
        
        return {
            'record_path': hea_path,
            'record_name': os.path.basename(hea_path),
            'fs': record.fs,
            'sig_len': record.sig_len,
            'n_sig': record.n_sig,
            'sig_name': record.sig_name,
            'dx_codes': dx_codes,
            'mapped_label': map_labels(dx_codes),
        }
    except Exception as e:
        print(f"Error parsing {hea_path}: {e}")
        return None

# Scan dataset
print("Scanning dataset...")
all_records = find_all_records(CFG.data_dir)
print(f"Found {len(all_records)} record files")

if len(all_records) == 0:
    print("\nWARNING: No records found! The dataset may not be downloaded.")
    print("Please download from: https://physionet.org/content/challenge-2020/1.0.2/")
    print("\nAttempting to download a subset...")
    # Try downloading at least one database
    import subprocess
    try:
        subprocess.run([
            "wget", "-r", "-N", "-c", "-np", "-nH", "--cut-dirs=3",
            "-P", CFG.data_dir,
            "https://physionet.org/files/challenge-2020/1.0.2/training/cpsc_2018/g1/"
        ], timeout=300, check=False)
        all_records = find_all_records(CFG.data_dir)
        print(f"After download attempt: {len(all_records)} records")
    except Exception as e:
        print(f"Download failed: {e}")


In [ ]:
# 4.2 Build metadata dataframe
metadata_list = []
for rec in all_records:
    info = parse_header(rec)
    if info:
        metadata_list.append(info)

df_meta = pd.DataFrame(metadata_list)
if len(df_meta) > 0:
    print(f"Successfully parsed {len(df_meta)} records")
    print(f"\nSampling frequencies: {df_meta['fs'].unique()}")
    print(f"Number of leads distribution: {df_meta['n_sig'].value_counts().to_dict()}")
    
    # Class distribution
    print("\n=== Class Distribution ===")
    class_counts = df_meta['mapped_label'].value_counts()
    for cls in CLASS_NAMES:
        count = class_counts.get(cls, 0)
        pct = 100 * count / len(df_meta) if len(df_meta) > 0 else 0
        print(f"  {cls:8s}: {count:6d} ({pct:5.1f}%)")
    
    # Filter to 12-lead records only
    df_meta = df_meta[df_meta['n_sig'] == 12].reset_index(drop=True)
    print(f"\n12-lead records: {len(df_meta)}")
    
    # Save labels
    labels_path = os.path.join(CFG.output_dir, "metadata", "labels_mapped.csv")
    df_meta[['record_name', 'mapped_label', 'fs', 'sig_len']].to_csv(labels_path, index=False)
    print(f"Labels saved to {labels_path}")
else:
    print("No records parsed. Check dataset path.")


## 5. Signal Preprocessing Pipeline

1. Load waveform using `wfdb`
2. Resample to 500 Hz
3. Re-order leads to standard 12-lead order: I, II, III, aVR, aVL, aVF, V1-V6
4. Bandpass filter (0.5-40 Hz)
5. Z-score normalization per lead
6. Fixed window (5000 samples = 10s @ 500Hz) with truncation/padding


In [ ]:
# 5.1 Signal preprocessing functions
STANDARD_LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

def load_ecg_signal(record_path, target_fs=500, seq_len=5000):
    """Load and preprocess a single ECG record."""
    try:
        record = wfdb.rdrecord(record_path)
        sig = record.p_signal  # (samples, leads)
        fs = record.fs
        lead_names = record.sig_name
        
        # Reorder to standard lead order
        lead_indices = []
        for std_lead in STANDARD_LEADS:
            found = False
            for i, name in enumerate(lead_names):
                if name.upper().replace(' ','') == std_lead.upper().replace(' ',''):
                    lead_indices.append(i)
                    found = True
                    break
            if not found:
                lead_indices.append(0)  # fallback
        
        sig = sig[:, lead_indices]
        
        # Resample if needed
        if fs != target_fs:
            num_samples = int(len(sig) * target_fs / fs)
            resampled = np.zeros((num_samples, 12))
            for ch in range(12):
                resampled[:, ch] = scipy_signal.resample(sig[:, ch], num_samples)
            sig = resampled
        
        # Bandpass filter (0.5-40 Hz)
        nyq = target_fs / 2
        low, high = 0.5 / nyq, 40.0 / nyq
        if high < 1.0:
            b, a = scipy_signal.butter(3, [low, high], btype='band')
            for ch in range(12):
                sig[:, ch] = scipy_signal.filtfilt(b, a, sig[:, ch])
        
        # Handle NaN/Inf
        sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Z-score per lead
        for ch in range(12):
            std = np.std(sig[:, ch])
            if std > 1e-8:
                sig[:, ch] = (sig[:, ch] - np.mean(sig[:, ch])) / std
        
        # Truncate or pad to seq_len
        if len(sig) >= seq_len:
            sig = sig[:seq_len, :]
        else:
            pad = np.zeros((seq_len - len(sig), 12))
            sig = np.vstack([sig, pad])
        
        return sig.astype(np.float32)  # (seq_len, 12)
    except Exception as e:
        return None

print("Signal preprocessing pipeline ready.")
print(f"  Target: {CFG.target_fs}Hz, {CFG.seq_len} samples ({CFG.seq_len/CFG.target_fs}s)")


## 6. Vision Preprocessing Pipeline

Using **Continuous Wavelet Transform (CWT)** scalograms as 2D image representations.
- Each lead produces a scalogram
- Combined into a multi-channel image
- Standardized to 224×224 RGB
- Cached to `cache/images/` for efficiency


In [ ]:
# 6.1 CWT Scalogram generation
def generate_scalogram(signal_1d, fs=500, img_size=224):
    """Generate CWT scalogram from a 1D signal."""
    scales = np.arange(1, 65)
    coeffs, freqs = pywt.cwt(signal_1d, scales, 'morl', sampling_period=1.0/fs)
    power = np.abs(coeffs) ** 2
    # Normalize
    if power.max() > 0:
        power = power / power.max()
    # Resize to img_size x img_size
    from PIL import Image
    img = Image.fromarray((power * 255).astype(np.uint8))
    img = img.resize((img_size, img_size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0

def ecg_to_image(ecg_signal, fs=500, img_size=224):
    """Convert 12-lead ECG to RGB image using CWT scalograms.
    Uses leads I, II, V1 as R, G, B channels."""
    # Select 3 representative leads for RGB
    lead_indices = [0, 1, 6]  # I, II, V1
    channels = []
    for idx in lead_indices:
        scalo = generate_scalogram(ecg_signal[:, idx], fs, img_size)
        channels.append(scalo)
    img = np.stack(channels, axis=0)  # (3, H, W)
    return img.astype(np.float32)

print("Vision preprocessing pipeline ready.")
print(f"  Output: {CFG.img_size}x{CFG.img_size} RGB scalogram images")


## 7. Train/Validation/Test Split

In [ ]:
# 7.1 Stratified split
if len(df_meta) > 0:
    # Encode labels
    label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
    idx_to_label = {i: name for i, name in enumerate(CLASS_NAMES)}
    df_meta['label_idx'] = df_meta['mapped_label'].map(label_to_idx)
    
    # Split: train/temp -> temp/val/test
    train_idx, temp_idx = train_test_split(
        np.arange(len(df_meta)),
        test_size=CFG.val_ratio + CFG.test_ratio,
        stratify=df_meta['label_idx'],
        random_state=CFG.seed
    )
    
    relative_test = CFG.test_ratio / (CFG.val_ratio + CFG.test_ratio)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=relative_test,
        stratify=df_meta.iloc[temp_idx]['label_idx'],
        random_state=CFG.seed
    )
    
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
    # Per-split class distribution
    for name, indices in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
        counts = df_meta.iloc[indices]['mapped_label'].value_counts()
        print(f"\n{name} distribution:")
        for cls in CLASS_NAMES:
            c = counts.get(cls, 0)
            print(f"  {cls}: {c}")
else:
    print("No data available for splitting.")
    train_idx, val_idx, test_idx = [], [], []


## 8. PyTorch Dataset Classes

In [ ]:
# 8.1 Signal Dataset
class ECGSignalDataset(Dataset):
    """Dataset for signal-based models (1D input)."""
    def __init__(self, df, indices, config):
        self.df = df.iloc[indices].reset_index(drop=True)
        self.config = config
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sig = load_ecg_signal(row['record_path'], self.config.target_fs, self.config.seq_len)
        if sig is None:
            sig = np.zeros((self.config.seq_len, 12), dtype=np.float32)
        # Transpose to (12, seq_len) for 1D Conv
        sig = sig.T
        label = row['label_idx']
        return torch.FloatTensor(sig), torch.LongTensor([label])[0]

# 8.2 Vision Dataset
class ECGImageDataset(Dataset):
    """Dataset for vision-based models (2D input)."""
    def __init__(self, df, indices, config):
        self.df = df.iloc[indices].reset_index(drop=True)
        self.config = config
        self.cache_dir = os.path.join(config.cache_dir, "images")
        os.makedirs(self.cache_dir, exist_ok=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = os.path.join(self.cache_dir, f"{row['record_name']}.npy")
        
        if os.path.exists(cache_path):
            img = np.load(cache_path)
        else:
            sig = load_ecg_signal(row['record_path'], self.config.target_fs, self.config.seq_len)
            if sig is None:
                sig = np.zeros((self.config.seq_len, 12), dtype=np.float32)
            img = ecg_to_image(sig, self.config.target_fs, self.config.img_size)
            np.save(cache_path, img)
        
        label = row['label_idx']
        return torch.FloatTensor(img), torch.LongTensor([label])[0]

# 8.3 Hybrid Dataset (returns both signal and image)
class ECGHybridDataset(Dataset):
    """Dataset for hybrid model (both 1D signal and 2D image)."""
    def __init__(self, df, indices, config):
        self.sig_ds = ECGSignalDataset(df, indices, config)
        self.img_ds = ECGImageDataset(df, indices, config)
    
    def __len__(self):
        return len(self.sig_ds)
    
    def __getitem__(self, idx):
        sig, label = self.sig_ds[idx]
        img, _ = self.img_ds[idx]
        return sig, img, label

print("Dataset classes ready.")


## 9. Model Definitions

### Model Architecture References

| Model | Type | Reference |
|-------|------|-----------|
| S1 | 1D ResNet | [ECG-Classification](https://github.com/Jazhyc/ECG-Classification) |
| S2 | 1D CNN Baseline | [ECG-Time-Series-Classification](https://github.com/Ertugrulmert/ECG-Time-Series-Classification) |
| V1 | 2D CNN (ResNet) | [ECG-Arrhythmia-Classification](https://github.com/21Rachit/ECG-Arrhythmia-Classification) |
| V2 | Vision Transformer | [Vision-Transformer-for-ECG-Classification](https://github.com/ShaileshKumar97/Vision-Transformer-for-ECG-Classification) |
| H1 | Hybrid (Signal+Vision) | [ECG-Multi-Label-Classification](https://github.com/JaeBinCHA7/ECG-Multi-Label-Classification-Using-Multi-Model) |


In [ ]:
# 9.1 Model S1 - 1D ResNet ECG Classifier
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 7, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch))
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

class ResNet1D(nn.Module):
    """1D ResNet for 12-lead ECG classification.
    Reference: https://github.com/Jazhyc/ECG-Classification
    Adapted for 12-lead input with residual 1D CNN backbone."""
    def __init__(self, num_classes=7, in_channels=12):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, 15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool1d(3, stride=2, padding=1)
        
        self.layer1 = nn.Sequential(ResBlock1D(64, 64), ResBlock1D(64, 64))
        self.layer2 = nn.Sequential(ResBlock1D(64, 128, 2), ResBlock1D(128, 128))
        self.layer3 = nn.Sequential(ResBlock1D(128, 256, 2), ResBlock1D(256, 256))
        self.layer4 = nn.Sequential(ResBlock1D(256, 512, 2), ResBlock1D(512, 512))
        
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        return self.fc(x)
    
    def get_embeddings(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.avgpool(x).squeeze(-1)

model_s1 = ResNet1D(NUM_CLASSES).to(DEVICE)
print(f"S1 (1D ResNet) params: {sum(p.numel() for p in model_s1.parameters()):,}")


In [ ]:
# 9.2 Model S2 - 1D CNN Baseline
class CNN1DBaseline(nn.Module):
    """Simple 1D CNN baseline for ECG classification.
    Reference: https://github.com/Ertugrulmert/ECG-Time-Series-Classification
    Double-conv blocks with batch norm and pooling."""
    def __init__(self, num_classes=7, in_channels=12):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(in_channels, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Block 2
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Block 3
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(4), nn.Dropout(0.2),
            # Block 4
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
    
    def forward(self, x):
        x = self.features(x).squeeze(-1)
        return self.classifier(x)
    
    def get_embeddings(self, x):
        return self.features(x).squeeze(-1)

model_s2 = CNN1DBaseline(NUM_CLASSES).to(DEVICE)
print(f"S2 (1D CNN Baseline) params: {sum(p.numel() for p in model_s2.parameters()):,}")


In [ ]:
# 9.3 Model V1 - 2D CNN on ECG Scalograms
class CNN2D(nn.Module):
    """2D CNN for ECG image classification using scalograms.
    Reference: https://github.com/21Rachit/ECG-Arrhythmia-Classification
    Simple ResNet-inspired 2D CNN backbone."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
    
    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.classifier(x)
    
    def get_embeddings(self, x):
        return self.features(x).flatten(1)

model_v1 = CNN2D(NUM_CLASSES).to(DEVICE)
print(f"V1 (2D CNN) params: {sum(p.numel() for p in model_v1.parameters()):,}")


In [ ]:
# 9.4 Model V2 - Vision Transformer for ECG
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim, patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        return x.flatten(2).transpose(1, 2)  # (B, n_patches, embed_dim)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Dropout(drop),
            nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(drop))
    
    def forward(self, x):
        x2 = self.norm1(x)
        x = x + self.attn(x2, x2, x2)[0]
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformerECG(nn.Module):
    """Vision Transformer for ECG scalogram classification.
    Reference: https://github.com/ShaileshKumar97/Vision-Transformer-for-ECG-Classification
    Lightweight ViT adapted for ECG image representations."""
    def __init__(self, img_size=224, patch_size=16, num_classes=7, 
                 embed_dim=192, depth=6, heads=4, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, heads, drop=drop) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        x = self.norm(x[:, 0])
        return self.head(x)
    
    def get_embeddings(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        return self.norm(x[:, 0])

model_v2 = VisionTransformerECG(CFG.img_size, num_classes=NUM_CLASSES).to(DEVICE)
print(f"V2 (ViT) params: {sum(p.numel() for p in model_v2.parameters()):,}")


In [ ]:
# 9.5 Model H1 - Late-Fusion Hybrid (Signal + Vision)
class HybridModel(nn.Module):
    """Late-fusion hybrid model combining signal and vision branches.
    Reference: https://github.com/JaeBinCHA7/ECG-Multi-Label-Classification-Using-Multi-Model
    
    Signal branch: 1D CNN/ResNet on raw 12-lead signal
    Vision branch: 2D CNN on ECG scalogram images
    Fusion: Concatenation of embeddings before final classifier
    """
    def __init__(self, num_classes=7):
        super().__init__()
        # Signal branch (simplified ResNet1D)
        self.signal_branch = nn.Sequential(
            nn.Conv1d(12, 32, 7, stride=2, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 5, stride=2, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 3, stride=2, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 3, stride=2, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        
        # Vision branch (simplified 2D CNN)
        self.vision_branch = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))
        
        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))
    
    def forward(self, signal, image):
        sig_emb = self.signal_branch(signal).squeeze(-1)  # (B, 256)
        vis_emb = self.vision_branch(image).flatten(1)     # (B, 256)
        fused = torch.cat([sig_emb, vis_emb], dim=1)       # (B, 512)
        return self.classifier(fused)
    
    def get_signal_embedding(self, signal):
        return self.signal_branch(signal).squeeze(-1)
    
    def get_vision_embedding(self, image):
        return self.vision_branch(image).flatten(1)

model_h1 = HybridModel(NUM_CLASSES).to(DEVICE)
print(f"H1 (Hybrid) params: {sum(p.numel() for p in model_h1.parameters()):,}")
print("\nAll 5 models defined successfully.")


## 10. Training Loops with CodeCarbon Tracking

Each model is trained with:
- Cross-entropy loss
- Adam optimizer (lr=1e-3)
- Early stopping (patience=5)
- CodeCarbon emissions tracking


In [ ]:
# 10.1 Training infrastructure
from codecarbon import EmissionsTracker

def train_epoch(model, loader, criterion, optimizer, device, is_hybrid=False):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        if is_hybrid:
            sig, img, labels = batch
            sig, img, labels = sig.to(device), img.to(device), labels.to(device)
            outputs = model(sig, img)
        else:
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * labels.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    
    return total_loss / max(total, 1), correct / max(total, 1)

def eval_epoch(model, loader, criterion, device, is_hybrid=False):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for batch in loader:
            if is_hybrid:
                sig, img, labels = batch
                sig, img, labels = sig.to(device), img.to(device), labels.to(device)
                outputs = model(sig, img)
            else:
                inputs, labels = batch
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
    
    return (total_loss / max(total, 1), correct / max(total, 1),
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

def train_model(model, train_loader, val_loader, model_name, config, is_hybrid=False):
    """Full training loop with CodeCarbon tracking."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    
    best_val_acc = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    # CodeCarbon tracking
    emissions_dir = os.path.join(config.output_dir, "emissions")
    tracker = EmissionsTracker(
        project_name="ecg-benchmark",
        experiment_id=model_name,
        output_dir=emissions_dir,
        log_level="error"
    )
    
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    start_time = time.time()
    tracker.start()
    
    for epoch in range(config.epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, is_hybrid)
        val_loss, val_acc, _, _, _ = eval_epoch(model, val_loader, criterion, DEVICE, is_hybrid)
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{config.epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            model_path = os.path.join(config.output_dir, "models", f"{model_name}_best.pt")
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    
    emissions = tracker.stop()
    train_time = time.time() - start_time
    
    print(f"  Best Val Acc: {best_val_acc:.4f}")
    print(f"  Training time: {train_time/60:.1f} min")
    print(f"  Emissions: {emissions:.6f} kg CO2eq" if emissions else "  Emissions: N/A")
    
    return history, best_val_acc, train_time, emissions or 0.0

print("Training infrastructure ready.")


In [ ]:
# 10.2 Create data loaders and train all models
if len(df_meta) > 0 and len(train_idx) > 0:
    # Signal loaders
    train_sig_ds = ECGSignalDataset(df_meta, train_idx, CFG)
    val_sig_ds = ECGSignalDataset(df_meta, val_idx, CFG)
    test_sig_ds = ECGSignalDataset(df_meta, test_idx, CFG)
    
    train_sig_loader = DataLoader(train_sig_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=0)
    val_sig_loader = DataLoader(val_sig_ds, batch_size=CFG.batch_size, num_workers=0)
    test_sig_loader = DataLoader(test_sig_ds, batch_size=CFG.batch_size, num_workers=0)
    
    # Image loaders
    train_img_ds = ECGImageDataset(df_meta, train_idx, CFG)
    val_img_ds = ECGImageDataset(df_meta, val_idx, CFG)
    test_img_ds = ECGImageDataset(df_meta, test_idx, CFG)
    
    train_img_loader = DataLoader(train_img_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=0)
    val_img_loader = DataLoader(val_img_ds, batch_size=CFG.batch_size, num_workers=0)
    test_img_loader = DataLoader(test_img_ds, batch_size=CFG.batch_size, num_workers=0)
    
    # Hybrid loaders
    train_hyb_ds = ECGHybridDataset(df_meta, train_idx, CFG)
    val_hyb_ds = ECGHybridDataset(df_meta, val_idx, CFG)
    test_hyb_ds = ECGHybridDataset(df_meta, test_idx, CFG)
    
    train_hyb_loader = DataLoader(train_hyb_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=0)
    val_hyb_loader = DataLoader(val_hyb_ds, batch_size=CFG.batch_size, num_workers=0)
    test_hyb_loader = DataLoader(test_hyb_ds, batch_size=CFG.batch_size, num_workers=0)
    
    print("Data loaders created.")
    
    # Train models
    results = {}
    
    # S1
    h, acc, t, e = train_model(model_s1, train_sig_loader, val_sig_loader, "S1_ResNet1D", CFG)
    results['S1'] = {'history': h, 'best_acc': acc, 'time': t, 'emissions': e}
    
    # S2
    h, acc, t, e = train_model(model_s2, train_sig_loader, val_sig_loader, "S2_CNN1D", CFG)
    results['S2'] = {'history': h, 'best_acc': acc, 'time': t, 'emissions': e}
    
    # V1
    h, acc, t, e = train_model(model_v1, train_img_loader, val_img_loader, "V1_CNN2D", CFG)
    results['V1'] = {'history': h, 'best_acc': acc, 'time': t, 'emissions': e}
    
    # V2
    h, acc, t, e = train_model(model_v2, train_img_loader, val_img_loader, "V2_ViT", CFG)
    results['V2'] = {'history': h, 'best_acc': acc, 'time': t, 'emissions': e}
    
    # H1
    h, acc, t, e = train_model(model_h1, train_hyb_loader, val_hyb_loader, "H1_Hybrid", CFG, is_hybrid=True)
    results['H1'] = {'history': h, 'best_acc': acc, 'time': t, 'emissions': e}
    
    print("\nAll models trained!")
else:
    print("No data available. Skipping training.")
    results = {}


## 11. Evaluation and Metrics

In [ ]:
# 11.1 Comprehensive evaluation
def evaluate_model(model, test_loader, model_name, config, is_hybrid=False):
    """Full evaluation with all required metrics."""
    criterion = nn.CrossEntropyLoss()
    
    # Load best weights
    model_path = os.path.join(config.output_dir, "models", f"{model_name}_best.pt")
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    
    _, acc, preds, labels, probs = eval_epoch(model, test_loader, criterion, DEVICE, is_hybrid)
    
    metrics = {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'macro_precision': precision_score(labels, preds, average='macro', zero_division=0),
        'macro_recall': recall_score(labels, preds, average='macro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'params': sum(p.numel() for p in model.parameters()),
    }
    
    # Per-class metrics
    report = classification_report(labels, preds, target_names=CLASS_NAMES, 
                                   output_dict=True, zero_division=0)
    for cls in CLASS_NAMES:
        if cls in report:
            metrics[f'{cls}_recall'] = report[cls]['recall']
            metrics[f'{cls}_precision'] = report[cls]['precision']
    
    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'Confusion Matrix - {model_name}')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')
    plt.tight_layout()
    fig.savefig(os.path.join(config.output_dir, "figures", f"cm_{model_name}.png"), dpi=150)
    plt.close()
    
    # AUROC (if enough classes present)
    try:
        metrics['auroc'] = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except:
        metrics['auroc'] = float('nan')
    
    # Inference time
    model.eval()
    sample = next(iter(test_loader))
    with torch.no_grad():
        start = time.time()
        for _ in range(10):
            if is_hybrid:
                model(sample[0][:1].to(DEVICE), sample[1][:1].to(DEVICE))
            else:
                model(sample[0][:1].to(DEVICE))
        metrics['inference_ms'] = (time.time() - start) / 10 * 1000
    
    return metrics, preds, labels, probs

# Run evaluation
if results:
    eval_results = {}
    
    models_info = [
        ('S1', model_s1, test_sig_loader, "S1_ResNet1D", False),
        ('S2', model_s2, test_sig_loader, "S2_CNN1D", False),
        ('V1', model_v1, test_img_loader, "V1_CNN2D", False),
        ('V2', model_v2, test_img_loader, "V2_ViT", False),
        ('H1', model_h1, test_hyb_loader, "H1_Hybrid", True),
    ]
    
    for key, model, loader, name, hyb in models_info:
        print(f"\nEvaluating {name}...")
        m, p, l, pr = evaluate_model(model, loader, name, CFG, hyb)
        eval_results[key] = m
        print(f"  Acc: {m['accuracy']:.4f}, Bal Acc: {m['balanced_accuracy']:.4f}, "
              f"F1: {m['macro_f1']:.4f}, AUROC: {m.get('auroc', 'N/A')}")
    
    print("\nEvaluation complete!")
else:
    eval_results = {}
    print("No trained models to evaluate.")


## 12. Benchmark Comparison Table

In [ ]:
# 12.1 Summary table
if eval_results:
    rows = []
    for key in ['S1', 'S2', 'V1', 'V2', 'H1']:
        m = eval_results.get(key, {})
        r = results.get(key, {})
        rows.append({
            'Model': key,
            'Type': 'Signal' if key.startswith('S') else ('Vision' if key.startswith('V') else 'Hybrid'),
            'Accuracy': f"{m.get('accuracy', 0):.4f}",
            'Balanced Acc': f"{m.get('balanced_accuracy', 0):.4f}",
            'Macro F1': f"{m.get('macro_f1', 0):.4f}",
            'AUROC': f"{m.get('auroc', 0):.4f}",
            'Params': f"{m.get('params', 0):,}",
            'Inference (ms)': f"{m.get('inference_ms', 0):.1f}",
            'Train Time (min)': f"{r.get('time', 0)/60:.1f}",
            'CO2 (kg)': f"{r.get('emissions', 0):.6f}",
        })
    
    df_results = pd.DataFrame(rows)
    print(df_results.to_string(index=False))
    
    # Save
    csv_path = os.path.join(CFG.output_dir, "metrics", "benchmark_comparison.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"\nSaved to {csv_path}")
else:
    print("No results to display.")


## 13. SHAP Explainability Analysis

Using SHAP to explain model predictions:
- **Signal models (S1, S2):** GradientExplainer for per-lead, per-time attributions
- **Vision models (V1, V2):** GradientExplainer for image attributions
- **Hybrid (H1):** Branch-specific SHAP analysis


In [ ]:
# 13.1 SHAP analysis
import shap

def shap_signal_analysis(model, test_loader, model_name, config, n_samples=20, n_background=50):
    """SHAP analysis for signal-based models."""
    model.eval()
    
    # Get background and test samples
    all_inputs, all_labels = [], []
    for inputs, labels in test_loader:
        all_inputs.append(inputs)
        all_labels.append(labels)
        if len(all_inputs) * inputs.shape[0] >= n_background + n_samples:
            break
    
    all_inputs = torch.cat(all_inputs)[:n_background + n_samples].to(DEVICE)
    all_labels = torch.cat(all_labels)[:n_background + n_samples]
    
    background = all_inputs[:n_background]
    test_data = all_inputs[n_background:n_background + n_samples]
    
    try:
        explainer = shap.GradientExplainer(model, background)
        shap_values = explainer.shap_values(test_data)
        
        if isinstance(shap_values, list):
            # Average across classes
            shap_avg = np.mean([np.abs(sv) for sv in shap_values], axis=0)
        else:
            shap_avg = np.abs(shap_values)
        
        # Per-lead importance
        lead_importance = np.mean(shap_avg, axis=(0, 2))  # Average over samples and time
        
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(STANDARD_LEADS, lead_importance)
        ax.set_title(f'SHAP Lead Importance - {model_name}')
        ax.set_ylabel('Mean |SHAP value|')
        plt.tight_layout()
        fig.savefig(os.path.join(config.output_dir, "figures/shap", f"lead_importance_{model_name}.png"), dpi=150)
        plt.close()
        
        # Time-lead heatmap for first sample
        if len(shap_avg) > 0:
            fig, ax = plt.subplots(figsize=(12, 4))
            sns.heatmap(shap_avg[0], ax=ax, yticklabels=STANDARD_LEADS,
                       cmap='RdBu_r', center=0)
            ax.set_title(f'SHAP Attribution Heatmap - {model_name} (Sample 0)')
            ax.set_xlabel('Time')
            plt.tight_layout()
            fig.savefig(os.path.join(config.output_dir, "figures/shap", f"heatmap_{model_name}.png"), dpi=150)
            plt.close()
        
        print(f"  SHAP analysis complete for {model_name}")
        return shap_values
    except Exception as e:
        print(f"  SHAP failed for {model_name}: {e}")
        return None

def shap_vision_analysis(model, test_loader, model_name, config, n_samples=10, n_background=30):
    """SHAP analysis for vision-based models."""
    model.eval()
    
    all_inputs, all_labels = [], []
    for inputs, labels in test_loader:
        all_inputs.append(inputs)
        all_labels.append(labels)
        if len(all_inputs) * inputs.shape[0] >= n_background + n_samples:
            break
    
    all_inputs = torch.cat(all_inputs)[:n_background + n_samples].to(DEVICE)
    background = all_inputs[:n_background]
    test_data = all_inputs[n_background:n_background + n_samples]
    
    try:
        explainer = shap.GradientExplainer(model, background)
        shap_values = explainer.shap_values(test_data)
        
        if isinstance(shap_values, list):
            shap_avg = np.mean([np.abs(sv) for sv in shap_values], axis=0)
        else:
            shap_avg = np.abs(shap_values)
        
        # Plot attribution heatmap for first few samples
        n_show = min(3, len(shap_avg))
        fig, axes = plt.subplots(n_show, 2, figsize=(12, 4*n_show))
        if n_show == 1:
            axes = [axes]
        
        for i in range(n_show):
            # Original image
            img = test_data[i].cpu().numpy().transpose(1, 2, 0)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            axes[i][0].imshow(img)
            axes[i][0].set_title(f'Input Image {i}')
            axes[i][0].axis('off')
            
            # SHAP heatmap
            attr = shap_avg[i].mean(axis=0)  # Average across channels
            axes[i][1].imshow(attr, cmap='hot')
            axes[i][1].set_title(f'SHAP Attribution {i}')
            axes[i][1].axis('off')
        
        plt.tight_layout()
        fig.savefig(os.path.join(config.output_dir, "figures/shap", f"vision_{model_name}.png"), dpi=150)
        plt.close()
        
        print(f"  SHAP analysis complete for {model_name}")
        return shap_values
    except Exception as e:
        print(f"  SHAP failed for {model_name}: {e}")
        return None

# Run SHAP
if eval_results:
    print("Running SHAP analyses...")
    shap_signal_analysis(model_s1, test_sig_loader, "S1_ResNet1D", CFG)
    shap_signal_analysis(model_s2, test_sig_loader, "S2_CNN1D", CFG)
    shap_vision_analysis(model_v1, test_img_loader, "V1_CNN2D", CFG)
    shap_vision_analysis(model_v2, test_img_loader, "V2_ViT", CFG)
    
    # H1 - Hybrid SHAP (signal branch)
    print("  Analyzing H1 signal branch...")
    shap_signal_analysis(model_h1.signal_branch, test_sig_loader, "H1_signal_branch", CFG)
    
    print("\nSHAP analysis complete. Figures saved to outputs/figures/shap/")
else:
    print("No models to analyze.")


### SHAP Interpretation

The SHAP attributions for ECG models should reveal:
- **Bundle branch blocks (RBBB/LBBB):** High attribution on QRS complexes, especially in leads V1-V3
- **Atrial fibrillation (AFIB):** Focus on P-wave regions and RR-interval variability
- **1st degree AV block:** Emphasis on PR interval regions
- **Normal:** More distributed, lower-magnitude attributions

These patterns align with clinical ECG interpretation guidelines, supporting the clinical plausibility of the models.


## 14. Carbon Emissions Summary (CodeCarbon)

In [ ]:
# 14.1 Emissions summary
if results:
    print("=" * 80)
    print("CARBON EMISSIONS SUMMARY")
    print("=" * 80)
    print(f"{'Model':<8} {'Type':<10} {'Train Time (min)':<20} {'Emissions (kg CO2eq)':<25} {'Notes'}")
    print("-" * 80)
    
    for key in ['S1', 'S2', 'V1', 'V2', 'H1']:
        r = results.get(key, {})
        model_type = 'Signal' if key.startswith('S') else ('Vision' if key.startswith('V') else 'Hybrid')
        t = r.get('time', 0) / 60
        e = r.get('emissions', 0)
        print(f"{key:<8} {model_type:<10} {t:<20.1f} {e:<25.6f}")
    
    total_emissions = sum(r.get('emissions', 0) for r in results.values())
    total_time = sum(r.get('time', 0) for r in results.values()) / 60
    print("-" * 80)
    print(f"{'TOTAL':<8} {'':<10} {total_time:<20.1f} {total_emissions:<25.6f}")
    print()
    print("These emissions estimates support AI sustainability reporting.")
    print("Signal-based models typically have lower computational cost than vision-based ones,")
    print("contributing to UN SDG 13 (Climate Action) considerations in AI deployment.")
else:
    print("No training results available.")


## 15. Research Interpretation

### Model Family Performance

**Signal-based models (S1, S2)** process raw 12-lead ECG signals directly, leveraging the temporal structure of cardiac electrical activity. These models are computationally efficient and can capture subtle waveform morphology changes.

**Vision-based models (V1, V2)** transform ECG signals into 2D representations (CWT scalograms), enabling the use of powerful image classification architectures. The ViT model (V2) uses self-attention mechanisms that can capture global patterns across the time-frequency representation.

**Hybrid model (H1)** combines both approaches through late fusion, potentially capturing complementary features from both signal and image domains.

### Key Trade-offs

| Factor | Signal Models | Vision Models | Hybrid |
|--------|--------------|---------------|--------|
| Computational Cost | Low | Medium-High | Highest |
| Training Time | Fast | Slower | Slowest |
| Carbon Emissions | Lowest | Medium | Highest |
| Interpretability | Direct (temporal) | Indirect (image) | Complex |
| Clinical Relevance | High (matches cardiologist workflow) | Medium | Medium-High |

### SHAP Clinical Plausibility

SHAP attributions should ideally emphasize:
- **QRS complexes** for bundle branch blocks — consistent with widened QRS morphology in RBBB/LBBB
- **P-wave absence/irregularity** for atrial fibrillation — matching the hallmark finding of AFIB
- **PR interval** for 1st degree AV block — reflecting prolonged atrioventricular conduction

### Deployment Recommendation

For a **cardiology-support setting**, the optimal model choice balances:
1. **Accuracy** — the model must reliably detect arrhythmias
2. **Interpretability** — clinicians need to understand model reasoning
3. **Efficiency** — real-time or near-real-time inference is critical
4. **Sustainability** — lower carbon footprint aligns with institutional ESG goals

Signal-based models (particularly S1 with its ResNet backbone) likely offer the best balance: strong accuracy with direct temporal interpretability, fast inference, and low computational overhead. The hybrid model may achieve marginally better accuracy but at significantly higher computational and environmental cost.

### Limitations

1. **Dataset scope:** The PhysioNet Challenge 2020 dataset, while large, represents specific clinical populations
2. **Label mapping:** Multi-label to single-label conversion via priority ordering may lose information
3. **Model simplification:** Architecture simplifications for feasibility may not capture full model potential
4. **SHAP approximation:** GradientExplainer provides approximate SHAP values; exact methods are computationally prohibitive
5. **Carbon tracking:** CodeCarbon estimates depend on hardware power models and grid carbon intensity data
